# 01 — FF-HEDM indexing on a small synthetic dataset

`midas-index` is a pure-Python / PyTorch FF-HEDM indexer — a
drop-in for the C `IndexerOMP`, with one device switch for
CPU / CUDA / MPS. This notebook runs the **Python backend on CPU**
over a small synthetic FF dataset that ships with the package's
tests (5 Cu grains, 4 rings).

By the end you will have:

1. Loaded a `paramstest.txt` with `Indexer.from_param_file`.
2. Loaded the binary observations (`Spots.bin`, `Data.bin`,
   `nData.bin`, `hkls.csv`, `SpotsToIndex.csv`).
3. Run the indexer on a few seed spots.
4. Read out per-seed orientation, position, and match statistics.

It runs in well under a minute on a laptop CPU.


In [1]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")  # torch+numba libomp coexist
import shutil, tempfile, time
from pathlib import Path

import numpy as np
import torch

from midas_index import Indexer
from midas_index import backend_c

print("torch:", torch.__version__, "| device: cpu")
# The bundled C binary is an optional accelerator. We probe it here but
# run the portable Python backend below regardless.
print("C backend (midas_indexer) available:", backend_c.available())


torch: 2.11.0 | device: cpu
C backend (midas_indexer) available: True


## 1. Stage the synthetic dataset

The package ships a tiny FF reference dataset under
`tests/data/ref_dataset/` (5 grains, 4 rings). We copy it into a
scratch workspace and point `OutputFolder` at a writable subdir.
The indexer reads `hkls.csv` by relative path, so we `chdir` into the
workspace.


In [2]:
import midas_index
PKG = Path(midas_index.__file__).resolve().parent
DATA = PKG.parent / "tests" / "data" / "ref_dataset"
assert DATA.exists(), f"reference dataset not found at {DATA}"

WORK = Path(tempfile.mkdtemp(prefix="midas_index_nb_"))
for f in ("paramstest.txt", "Spots.bin", "Data.bin", "nData.bin",
          "hkls.csv", "SpotsToIndex.csv"):
    shutil.copy2(DATA / f, WORK / f)
(WORK / "out").mkdir()

# Re-point OutputFolder at our scratch dir.
import re
ptxt = (WORK / "paramstest.txt").read_text()
ptxt = re.sub(r"OutputFolder .*", f"OutputFolder {WORK}/out", ptxt)
(WORK / "paramstest.txt").write_text(ptxt)

os.chdir(WORK)
print("workspace:", WORK)
print()
print((WORK / "paramstest.txt").read_text())


workspace: /var/folders/qw/k6gzh2ws7w397493kq4vnl_w0001pb/T/midas_index_nb_z_gjmhtt

Wavelength 0.172979
Distance 1000000.0
px 200.0
SpaceGroup 225
LatticeConstant 4.08 4.08 4.08 90.0 90.0 90.0
Rsample 250.0
Hbeam 200.0
StepsizePos 100.0
StepsizeOrient 0.5
MarginOme 0.5
MarginRadius 500.0
MarginRadial 500.0
MarginEta 500.0
EtaBinSize 0.1
OmeBinSize 0.1
ExcludePoleAngle 6.0
MinMatchesToAcceptFrac 0.1
OmegaRange -180 180
BoxSize -2000000 2000000 -2000000 2000000
UseFriedelPairs 0
OutputFolder /var/folders/qw/k6gzh2ws7w397493kq4vnl_w0001pb/T/midas_index_nb_z_gjmhtt/out
RingNumbers 1
RingNumbers 2
RingNumbers 3
RingNumbers 4
RingRadii 73582.31550724161
RingRadii 85023.04143552633
RingRadii 120567.4304605822
RingRadii 141666.90427076322



## 2. Build the indexer and load observations

`Indexer.from_param_file` parses `paramstest.txt` into typed
`IndexerParams`. `load_observations(cwd=".")` reads the binary spot
table, the (Data/nData) spatial bins, the hkls, and the list of seed
spot IDs.


In [3]:
indexer = Indexer.from_param_file("paramstest.txt", device="cpu", dtype="float64")
indexer.load_observations(cwd=".")

obs = indexer._observations
print("observed spots :", obs["spots"].shape)
print("hkls (real)    :", obs["hkls_real"].shape)
print("seed spot ids  :", obs["spot_ids"][:10], "...")
print("ring numbers   :", indexer.params.RingNumbers)
print("space group    :", indexer.params.SpaceGroup,
      "| lattice:", indexer.params.LatticeConstant)


observed spots : (9920, 9)
hkls (real)    : (50, 7)
seed spot ids  : [ 1  2  3  4  5  6  7  8 49 50] ...
ring numbers   : [1, 2, 3, 4]
space group    : 225 | lattice: (4.08, 4.08, 4.08, 90.0, 90.0, 90.0)


## 3. Run the indexer

`run()` evaluates each seed spot: it enumerates candidate
orientations, forward-simulates their theoretical spots, matches
against the observed bins, and keeps the best (orientation, position)
tuple. We index the first few seeds to keep it fast; drop
`n_spots_to_index` to index them all.


In [4]:
t0 = time.time()
result = indexer.run(
    block_nr=0, n_blocks=1,
    n_spots_to_index=5,    # first 5 seeds; None = all
    num_procs=1,
    backend="python",      # portable path. "c-omp" uses the bundled binary.
)
print(f"indexed {len(result.seeds)} seeds in {time.time() - t0:.2f} s")


/Users/hsharma/opt/MIDAS/packages/midas_index/midas_index/pipeline.py:98: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:219.)
  self.obs = torch.as_tensor(np.asarray(obs), device=device, dtype=dtype)


indexed 5 seeds in 0.23 s


## 4. Inspect the recovered grains

Each `SeedResult` carries the best orientation matrix, the sample-frame
position, and the matched-spot bookkeeping (`n_matches` of `n_t_spots`
theoretical spots, mean internal angle `avg_ia`).


In [5]:
print(f"{'spot':>5} {'n_match':>7} {'n_theor':>7} {'frac':>6} {'avg_ia(deg)':>11}")
for s in result.seeds:
    print(f"{s.spot_id:5d} {s.n_matches:7d} {s.n_t_spots:7d} "
          f"{s.frac_matches:6.2f} {float(s.avg_ia):11.4f}")

if result.seeds:
    s0 = result.seeds[0]
    print()
    print("seed", s0.spot_id, "best orientation matrix (row-major 3x3):")
    print(np.asarray(s0.best_or_mat).reshape(3, 3))
    print("sample-frame position (µm):", np.asarray(s0.best_pos))


 spot n_match n_theor   frac avg_ia(deg)
    1      16      96   0.17      0.1619
    2      16      96   0.17      0.0518
    3      16      96   0.17      0.0901
    4      16      96   0.17      0.0833
    5      16      96   0.17      0.1663

seed 1 best orientation matrix (row-major 3x3):
[[ 0.57369016 -0.10480832  0.8123391 ]
 [-0.77809325 -0.37951355  0.50054007]
 [ 0.25583293 -0.91923048 -0.29927383]]
sample-frame position (µm): [ 88.2871029  179.88470829   7.96996557]


## Notes

- **Device portability**: pass `device="cuda"` or `device="mps"` to
  `from_param_file`; everything else is identical. (This machine is
  CPU-only.)
- **C backend**: `run(..., backend="c-omp")` shells out to the bundled
  `midas_indexer` binary (when `backend_c.available()` is `True`),
  writing consolidated `IndexBest_all.bin` to `OutputFolder`. The
  Python backend is bit-identical to C `IndexerOMP` on the parity gate.
- **CLI**: the same run is `midas-index paramstest.txt 0 1 1000 8`.

See [02 — soft beam attribution](02_soft_attribution.ipynb) and
[03 — scanning / PF indexing](03_scanning_pf.ipynb).
